# 01 — Bahrain 2024: driver-lap table (saved CSV)

**Goal:** build one race’s wide driver-lap table and write it under `data/processed/`.

**Not yet:** `process_race()` for all races, or model training.

Schema: `docs/DRIVER_LAP_SCHEMA.md`

In [61]:
"""Load 2024 Bahrain laps as an identifier spine DataFrame."""

from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

# Repo root: works if CWD is repo root or notebooks/
CWD = Path.cwd()
ROOT = CWD if (CWD / "data" / "raw").exists() else CWD.parent

SESSION_DIR = ROOT / "data" / "raw" / "2024" / "sessions" / "1229_9472_bahrain_race"
LAPS_PATH = SESSION_DIR / "laps.json"

# Season is known from the folder year (not stored inside laps.json).
SEASON = 2024

# --- Load raw laps list from OpenF1 download ---
raw_laps = json.loads(LAPS_PATH.read_text())
laps = pd.DataFrame(raw_laps)

# --- Keep only identifier-style columns for this step ---
ID_COLS = ["meeting_key", "session_key", "driver_number", "lap_number", "date_start"]
spine = laps[ID_COLS].copy()
spine.insert(0, "season", SEASON)

# Stable order: easier to inspect one driver at a time
spine = spine.sort_values(["driver_number", "lap_number"]).reset_index(drop=True)

print("shape:", spine.shape)
print("columns:", list(spine.columns))
spine.head(10)

shape: (1129, 6)
columns: ['season', 'meeting_key', 'session_key', 'driver_number', 'lap_number', 'date_start']


,season,meeting_key,session_key,driver_number,lap_number,date_start
0,2024,1229,9472,1,1,2024-03-02T15:03:42.341000+00:00
1,2024,1229,9472,1,2,2024-03-02T15:05:20.099000+00:00
2,2024,1229,9472,1,3,2024-03-02T15:06:56.378000+00:00
3,2024,1229,9472,1,4,2024-03-02T15:08:33.130000+00:00
4,2024,1229,9472,1,5,2024-03-02T15:10:09.704000+00:00
5,2024,1229,9472,1,6,2024-03-02T15:11:46.978000+00:00
6,2024,1229,9472,1,7,2024-03-02T15:13:24.080000+00:00
7,2024,1229,9472,1,8,2024-03-02T15:15:01.132000+00:00
8,2024,1229,9472,1,9,2024-03-02T15:16:38.044000+00:00
9,2024,1229,9472,1,10,2024-03-02T15:18:15.404000+00:00


In [62]:
# Quick sanity checks — expect ~20 drivers and lap numbers starting at 1
print("unique drivers:", spine["driver_number"].nunique())
print("lap_number range:", spine["lap_number"].min(), "→", spine["lap_number"].max())
print("nulls per column:\n", spine.isna().sum())

# One driver sample
spine[spine["driver_number"] == 1].head(5)

unique drivers: 20
lap_number range: 1 → 57
nulls per column:
 season           0
meeting_key      0
session_key      0
driver_number    0
lap_number       0
date_start       0
dtype: int64


,season,meeting_key,session_key,driver_number,lap_number,date_start
0,2024,1229,9472,1,1,2024-03-02T15:03:42.341000+00:00
1,2024,1229,9472,1,2,2024-03-02T15:05:20.099000+00:00
2,2024,1229,9472,1,3,2024-03-02T15:06:56.378000+00:00
3,2024,1229,9472,1,4,2024-03-02T15:08:33.130000+00:00
4,2024,1229,9472,1,5,2024-03-02T15:10:09.704000+00:00


## Join stints

Each stint covers a lap range for one driver: `lap_start` … `lap_end`.

For a spine row at `lap_number`, we keep the stint where:

```text
lap_start  ≤  lap_number  ≤  lap_end
```

Then:

```text
tyre_age     = tyre_age_at_start + (lap_number - lap_start)
stint_length = (lap_number - lap_start) + 1
```

In [63]:
# --- Load stints for the same session ---
stints = pd.DataFrame(json.loads((SESSION_DIR / "stints.json").read_text()))

print("stints shape:", stints.shape)
print("stint columns:", list(stints.columns))
stints[stints["driver_number"] == 1].sort_values("stint_number")


stints shape: (63, 8)
stint columns: ['meeting_key', 'session_key', 'stint_number', 'driver_number', 'lap_start', 'lap_end', 'compound', 'tyre_age_at_start']


,meeting_key,session_key,stint_number,driver_number,lap_start,lap_end,compound,tyre_age_at_start
0,1229,9472,1,1,1,17,SOFT,3
39,1229,9472,2,1,18,37,HARD,0
58,1229,9472,3,1,38,57,SOFT,0


In [64]:
# --- Attach the stint that covers each lap ---
# Merge on driver (+ meeting/session keys), then keep rows where lap is inside the stint window.
# This briefly creates laps×stints per driver, then the filter reduces it back to one row per lap.

merged = spine.merge(
    stints[
        [
            "meeting_key",
            "session_key",
            "driver_number",
            "stint_number",
            "lap_start",
            "lap_end",
            "compound",
            "tyre_age_at_start",
        ]
    ],
    on=["meeting_key", "session_key", "driver_number"],
    how="left",
)

# Keep only the stint whose lap range contains this lap_number.
in_stint = (merged["lap_number"] >= merged["lap_start"]) & (
    merged["lap_number"] <= merged["lap_end"]
)
with_stints = merged.loc[in_stint].copy()

# Schema names from DRIVER_LAP_SCHEMA.md
with_stints = with_stints.rename(
    columns={
        "compound": "current_compound",
        "stint_number": "current_stint_number",
    }
)

# Tyre age grows by one each lap into the stint (OpenF1 gives age at stint start).
with_stints["tyre_age"] = (
    with_stints["tyre_age_at_start"]
    + (with_stints["lap_number"] - with_stints["lap_start"])
)

# How long this stint has been running as of this lap (inclusive).
with_stints["stint_length"] = (
    with_stints["lap_number"] - with_stints["lap_start"]
) + 1

# Drop helper columns we only needed for the join math.
with_stints = with_stints.drop(columns=["lap_start", "lap_end", "tyre_age_at_start"])

with_stints = with_stints.sort_values(
    ["driver_number", "lap_number"]
).reset_index(drop=True)

print("spine rows:", len(spine), "| after stint join:", len(with_stints))
print("columns:", list(with_stints.columns))
with_stints.head(10)


spine rows: 1129 | after stint join: 1129
columns: ['season', 'meeting_key', 'session_key', 'driver_number', 'lap_number', 'date_start', 'current_stint_number', 'current_compound', 'tyre_age', 'stint_length']


,season,meeting_key,session_key,driver_number,lap_number,date_start,current_stint_number,current_compound,tyre_age,stint_length
0,2024,1229,9472,1,1,2024-03-02T15:03:42.341000+00:00,1,SOFT,3,1
1,2024,1229,9472,1,2,2024-03-02T15:05:20.099000+00:00,1,SOFT,4,2
2,2024,1229,9472,1,3,2024-03-02T15:06:56.378000+00:00,1,SOFT,5,3
3,2024,1229,9472,1,4,2024-03-02T15:08:33.130000+00:00,1,SOFT,6,4
4,2024,1229,9472,1,5,2024-03-02T15:10:09.704000+00:00,1,SOFT,7,5
5,2024,1229,9472,1,6,2024-03-02T15:11:46.978000+00:00,1,SOFT,8,6
6,2024,1229,9472,1,7,2024-03-02T15:13:24.080000+00:00,1,SOFT,9,7
7,2024,1229,9472,1,8,2024-03-02T15:15:01.132000+00:00,1,SOFT,10,8
8,2024,1229,9472,1,9,2024-03-02T15:16:38.044000+00:00,1,SOFT,11,9
9,2024,1229,9472,1,10,2024-03-02T15:18:15.404000+00:00,1,SOFT,12,10


In [65]:
# Sanity: every spine lap should match exactly one stint (no dupes, no gaps).
print("duplicate driver-lap rows:", with_stints.duplicated(["driver_number", "lap_number"]).sum())
print("laps missing a stint:", len(spine) - len(with_stints))
print("null compounds:", with_stints["current_compound"].isna().sum())

# Driver 1: soft → hard → soft; stint boundaries should match stints table (ends 17, 37, 57).
with_stints.loc[
    with_stints["driver_number"] == 1,
    [
        "lap_number",
        "current_stint_number",
        "current_compound",
        "tyre_age",
        "stint_length",
    ],
].iloc[[0, 16, 17, 36, 37, 56]]


duplicate driver-lap rows: 0
laps missing a stint: 0
null compounds: 0


,lap_number,current_stint_number,current_compound,tyre_age,stint_length
0,1,1,SOFT,3,1
16,17,1,SOFT,19,17
17,18,2,HARD,0,1
36,37,2,HARD,19,20
37,38,3,SOFT,0,1
56,57,3,SOFT,19,20


## Join pits

`pit.json` lists each stop with a `lap_number` (lap on which the stop happened).

For driver 1 in Bahrain: pits on laps **17** and **37** (new stints start 18 and 38).

**Conventions for this step** (at the end of completed lap `L`):

```text
number_of_pit_stops  = how many pits with pit_lap ≤ L
laps_since_last_pit  = L - last_pit_lap   (if any pit ≤ L)
                     = L                  (if no pit yet — treat race start as reference)
```

We are **not** building `pit_within_3/5/7` labels yet — that is a later step.


In [66]:
# --- Load pit stops for this session ---
pits = pd.DataFrame(json.loads((SESSION_DIR / "pit.json").read_text()))

print("pit rows:", len(pits))
print("columns:", list(pits.columns))
pits[pits["driver_number"] == 1].sort_values("lap_number")


pit rows: 43
columns: ['date', 'session_key', 'lap_number', 'lane_duration', 'stop_duration', 'pit_duration', 'meeting_key', 'driver_number']


,date,session_key,lap_number,lane_duration,stop_duration,pit_duration,meeting_key,driver_number
19,2024-03-02T15:31:37.825000+00:00,9472,17,25.0,None,25.0,1229,1
38,2024-03-02T16:03:54.529000+00:00,9472,37,24.2,None,24.2,1229,1


In [67]:
# --- Add pit-count features onto the stint-joined table ---
# Start from with_stints (must re-run earlier cells first).


def add_pit_features(driver_laps: pd.DataFrame, pits: pd.DataFrame) -> pd.DataFrame:
    """
    Attach number_of_pit_stops and laps_since_last_pit to each driver-lap row.

    For each row at lap L, only pits with pit_lap <= L are visible (no future pits).
    """
    out = driver_laps.copy()
    stop_counts: list[int] = []
    laps_since: list[int] = []

    # Group pits by driver once for faster lookup.
    pits_by_driver = {
        driver: group["lap_number"].sort_values().to_numpy()
        for driver, group in pits.groupby("driver_number")
    }

    for row in out.itertuples(index=False):
        driver = row.driver_number
        lap = row.lap_number
        driver_pits = pits_by_driver.get(driver, [])

        # Pits that have already happened as of this completed lap.
        past = [p for p in driver_pits if p <= lap]
        stop_counts.append(len(past))

        if past:
            # Laps elapsed since the most recent stop (0 on the pit lap itself).
            laps_since.append(lap - past[-1])
        else:
            # No stop yet: count from race start (lap 1 → 1, lap 10 → 10).
            laps_since.append(lap)

    out["number_of_pit_stops"] = stop_counts
    out["laps_since_last_pit"] = laps_since
    return out


with_pits = add_pit_features(with_stints, pits)

print("rows:", len(with_pits))
with_pits.loc[
    with_pits["driver_number"] == 1,
    [
        "lap_number",
        "current_compound",
        "tyre_age",
        "number_of_pit_stops",
        "laps_since_last_pit",
    ],
].iloc[[15, 16, 17, 35, 36, 37]]


rows: 1129


,lap_number,current_compound,tyre_age,number_of_pit_stops,laps_since_last_pit
15,16,SOFT,18,0,16
16,17,SOFT,19,1,0
17,18,HARD,0,1,1
35,36,HARD,18,1,19
36,37,HARD,19,2,0
37,38,SOFT,0,2,1


## Join position (as-of)

`position.json` is a **time series** of place changes (not one row per lap).

For each driver-lap we want place **at the end of that lap**:

```text
as_of = date_start + lap_duration
current_position = latest position row with date ≤ as_of  (same driver)
```

Tool: `pd.merge_asof(..., direction="backward")`.

Driver 1 finished P1 every lap here — use **driver 14** to see real changes.


In [68]:
# --- Load position time series for this session ---
positions = pd.DataFrame(json.loads((SESSION_DIR / "position.json").read_text()))

print("position rows:", len(positions))
print("columns:", list(positions.columns))
# Driver 14 moves around a lot — useful for a later spot-check.
positions[positions["driver_number"] == 14].sort_values("date").head(8)


position rows: 698
columns: ['date', 'session_key', 'driver_number', 'position', 'meeting_key']


,date,session_key,driver_number,position,meeting_key
5,2024-03-02T14:03:47.739000+00:00,9472,14,6,1229
25,2024-03-02T15:03:49.260000+00:00,9472,14,7,1229
32,2024-03-02T15:03:49.419000+00:00,9472,14,8,1229
39,2024-03-02T15:03:49.429000+00:00,9472,14,9,1229
46,2024-03-02T15:03:49.741000+00:00,9472,14,10,1229
54,2024-03-02T15:03:49.843000+00:00,9472,14,11,1229
62,2024-03-02T15:03:49.937000+00:00,9472,14,12,1229
70,2024-03-02T15:03:50.137000+00:00,9472,14,13,1229


In [69]:
# --- As-of join: position at end of each completed lap ---
# Needs with_pits from earlier cells, and `laps` (still in memory) for lap_duration.


def add_current_position(driver_laps: pd.DataFrame, positions: pd.DataFrame, laps: pd.DataFrame) -> pd.DataFrame:
    """Attach current_position via merge_asof at end-of-lap time (no future rows)."""
    # Bring lap_duration so we can approximate when this lap finished.
    out = driver_laps.merge(
        laps[["driver_number", "lap_number", "lap_duration"]],
        on=["driver_number", "lap_number"],
        how="left",
    )

    # Parse timestamps (OpenF1 mixes with/without fractional seconds).
    out["date_start"] = pd.to_datetime(out["date_start"], utc=True, format="ISO8601")
    # Missing duration → treat as 0 so as_of falls back to date_start.
    out["as_of"] = out["date_start"] + pd.to_timedelta(out["lap_duration"].fillna(0), unit="s")

    pos = positions.copy()
    pos["date"] = pd.to_datetime(pos["date"], utc=True, format="ISO8601")
    pos = pos[["driver_number", "date", "position"]].sort_values("date")

    # Latest known place at or before as_of, matched per driver.
    joined = pd.merge_asof(
        out.sort_values("as_of"),
        pos,
        left_on="as_of",
        right_on="date",
        by="driver_number",
        direction="backward",
    )
    joined = joined.rename(columns={"position": "current_position"})
    # Drop helper timestamp from the position stream (keep as_of for debugging if useful).
    return joined.drop(columns=["date"]).sort_values(["driver_number", "lap_number"]).reset_index(drop=True)


with_pos = add_current_position(with_pits, positions, laps)

print("rows:", len(with_pos), "null positions:", with_pos["current_position"].isna().sum())
with_pos.loc[
    with_pos["driver_number"] == 14,
    ["lap_number", "current_position", "number_of_pit_stops", "laps_since_last_pit"],
].iloc[[0, 2, 4, 9, 11, 12, 13, 14, 15]]


rows: 1129 null positions: 0


,lap_number,current_position,number_of_pit_stops,laps_since_last_pit
338,1,6,0,1
340,3,7,0,3
342,5,8,0,5
347,10,9,0,10
349,12,6,0,12
350,13,4,0,13
351,14,3,0,14
352,15,2,1,0
353,16,10,1,1


## Join intervals (as-of)

Same idea as position: `intervals.json` is a time series.

For each driver-lap (using the same end-of-lap `as_of` already on `with_pos`):

```text
gap_to_leader  = latest intervals.gap_to_leader  with date ≤ as_of
interval_ahead = latest intervals.interval       with date ≤ as_of
  (gap to car immediately ahead; OpenF1 name is just "interval")
```

**Quirk:** when a car is lapped, OpenF1 may store `gap_to_leader` as `"+1 LAP"` / `"+2 LAPS"` (strings), not seconds. We keep that as-is for now.


In [70]:
# --- Load interval time series for this session ---
intervals = pd.DataFrame(json.loads((SESSION_DIR / "intervals.json").read_text()))

print("interval rows:", len(intervals))
print("columns:", list(intervals.columns))
intervals[intervals["driver_number"] == 14].sort_values("date").head(5)


interval rows: 29844
columns: ['date', 'session_key', 'gap_to_leader', 'meeting_key', 'driver_number', 'interval']


,date,session_key,gap_to_leader,meeting_key,driver_number,interval
16,2024-03-02T15:03:49.238000+00:00,9472,0.737,1229,14,0.240
36,2024-03-02T15:03:51.234000+00:00,9472,0.643,1229,14,0.093
55,2024-03-02T15:03:53.327000+00:00,9472,0.72,1229,14,0.011
74,2024-03-02T15:03:58.348000+00:00,9472,1.135,1229,14,0.004
92,2024-03-02T15:04:02.732000+00:00,9472,1.414,1229,14,0.145


In [71]:
# --- As-of join: gaps at end of each completed lap ---
# Reuses `as_of` from with_pos (same end-of-lap clock as current_position).


def add_interval_features(driver_laps: pd.DataFrame, intervals: pd.DataFrame) -> pd.DataFrame:
    """Attach gap_to_leader and interval_ahead via merge_asof (no future rows)."""
    out = driver_laps.copy()
    # Guard: position step must have built as_of already.
    if "as_of" not in out.columns:
        raise ValueError("expected as_of on driver_laps — run the position join first")

    iv = intervals.copy()
    iv["date"] = pd.to_datetime(iv["date"], utc=True, format="ISO8601")
    iv = iv[["driver_number", "date", "gap_to_leader", "interval"]].sort_values("date")

    joined = pd.merge_asof(
        out.sort_values("as_of"),
        iv,
        left_on="as_of",
        right_on="date",
        by="driver_number",
        direction="backward",
    )
    # Rename OpenF1 "interval" to the clearer schema name.
    joined = joined.rename(columns={"interval": "interval_ahead"})
    return joined.drop(columns=["date"]).sort_values(["driver_number", "lap_number"]).reset_index(drop=True)


with_intervals = add_interval_features(with_pos, intervals)

print("rows:", len(with_intervals))
print(
    "null gap / interval:",
    with_intervals["gap_to_leader"].isna().sum(),
    with_intervals["interval_ahead"].isna().sum(),
)
with_intervals.loc[
    with_intervals["driver_number"] == 14,
    ["lap_number", "current_position", "gap_to_leader", "interval_ahead"],
].iloc[[0, 4, 9, 11, 14, 15, 16]]


rows: 1129
null gap / interval: 20 13


,lap_number,current_position,gap_to_leader,interval_ahead
338,1,6,4.395,1.059
342,5,8,11.66,0.894
347,10,9,18.702,0.938
349,12,6,21.669,1.176
352,15,2,26.437,26.437
353,16,10,50.546,0.615
354,17,9,47.62,5.705


## Join weather (as-of)

`weather.json` is also a time series, but **session-wide** (no `driver_number`).

Same end-of-lap `as_of` clock. One weather snapshot is attached to **every** driver on that lap:

```text
latest weather row with date ≤ as_of
→ air_temperature, track_temperature, humidity, rainfall, wind_speed
```

Bahrain 2024 was dry (`rainfall` stays 0); track temp drifts down a few degrees over the race.


In [72]:
# --- Load session weather time series ---
weather = pd.DataFrame(json.loads((SESSION_DIR / "weather.json").read_text()))

print("weather rows:", len(weather))
print("columns:", list(weather.columns))
weather[["date", "air_temperature", "track_temperature", "humidity", "rainfall", "wind_speed"]].head(3)


weather rows: 157
columns: ['date', 'session_key', 'air_temperature', 'pressure', 'humidity', 'wind_direction', 'meeting_key', 'rainfall', 'wind_speed', 'track_temperature']


,date,air_temperature,track_temperature,humidity,rainfall,wind_speed
0,2024-03-02T14:03:56.523000+00:00,18.9,26.5,46.0,0,0.9
1,2024-03-02T14:04:56.514000+00:00,18.9,26.5,46.0,0,1.0
2,2024-03-02T14:05:56.523000+00:00,18.9,26.5,46.0,0,1.0


In [73]:
# --- As-of join: weather at end of each completed lap (same for all drivers) ---


def add_weather_features(driver_laps: pd.DataFrame, weather: pd.DataFrame) -> pd.DataFrame:
    """Attach schema weather columns via merge_asof on as_of (no driver key)."""
    out = driver_laps.copy()
    if "as_of" not in out.columns:
        raise ValueError("expected as_of on driver_laps — run the position join first")

    # Only the schema fields we care about for v1.
    cols = [
        "air_temperature",
        "track_temperature",
        "humidity",
        "rainfall",
        "wind_speed",
    ]
    wx = weather.copy()
    wx["date"] = pd.to_datetime(wx["date"], utc=True, format="ISO8601")
    wx = wx[["date", *cols]].sort_values("date")

    # No `by=` — weather is global to the session, not per driver.
    joined = pd.merge_asof(
        out.sort_values("as_of"),
        wx,
        left_on="as_of",
        right_on="date",
        direction="backward",
    )
    return joined.drop(columns=["date"]).sort_values(["driver_number", "lap_number"]).reset_index(drop=True)


with_weather = add_weather_features(with_intervals, weather)

print("rows:", len(with_weather))
print("track_temperature range:", with_weather["track_temperature"].min(), "→", with_weather["track_temperature"].max())
with_weather.loc[
    with_weather["driver_number"] == 14,
    ["lap_number", "air_temperature", "track_temperature", "humidity", "rainfall", "wind_speed"],
].iloc[[0, 19, 39, 56]]


rows: 1129
track_temperature range: 21.9 → 23.8


,lap_number,air_temperature,track_temperature,humidity,rainfall,wind_speed
338,1,18.3,23.8,49.0,0,0.9
357,20,18.1,23.2,50.0,0,0.7
377,40,17.9,22.4,50.0,0,0.4
394,57,17.6,21.9,51.0,0,0.7


## Join race control (derived flags)

`race_control.json` is an event log (messages + flags), not a ready-made “SC on” column.

**v1 rules** (walk events in time, then as-of join on `as_of`):

| Flag | Turns **on** | Turns **off** |
|------|----------------|---------------|
| `safety_car_active` | message contains `SAFETY CAR DEPLOYED` (not virtual) | `SAFETY CAR IN THIS LAP` |
| `virtual_safety_car_active` | `VIRTUAL SAFETY CAR DEPLOYED` / `VSC DEPLOYED` | `…ENDING` / `VSC ENDING` |
| `yellow_flag_active` | flag `YELLOW` or `DOUBLE YELLOW` | flag `CLEAR` / `GREEN` / `CHEQUERED` |
| `red_flag_recent` | any `RED FLAG` in the **5 minutes** before `as_of` | (window expires) |

Bahrain 2024: **no SC/VSC/red** — expect a brief yellow around **lap 10** for the leader.

Yellow rule is simplified (any CLEAR clears yellow globally; fine for v1).


In [74]:
# --- Load race-control event log ---
race_control = pd.DataFrame(json.loads((SESSION_DIR / "race_control.json").read_text()))

print("race_control rows:", len(race_control))
# Peek at flag-related rows only (most messages are noise for our flags).
race_control.loc[
    race_control["flag"].isin(["YELLOW", "DOUBLE YELLOW", "CLEAR", "GREEN", "RED", "CHEQUERED"])
    | race_control["message"].fillna("").str.contains("SAFETY CAR|VSC|RED FLAG", case=False),
    ["date", "flag", "message"],
]


race_control rows: 71


,date,flag,message
1,2024-03-02T14:20:01+00:00,GREEN,GREEN LIGHT - PIT EXIT OPEN
5,2024-03-02T15:03:42+00:00,GREEN,GREEN LIGHT - PIT EXIT OPEN
7,2024-03-02T15:03:59+00:00,DOUBLE YELLOW,DOUBLE YELLOW IN TRACK SECTOR 2
8,2024-03-02T15:04:03+00:00,CLEAR,CLEAR IN TRACK SECTOR 2
9,2024-03-02T15:04:29+00:00,DOUBLE YELLOW,DOUBLE YELLOW IN TRACK SECTOR 2
10,2024-03-02T15:04:36+00:00,CLEAR,CLEAR IN TRACK SECTOR 2
14,2024-03-02T15:05:55+00:00,CLEAR,CLEAR IN TRACK SECTOR 4
20,2024-03-02T15:19:07+00:00,YELLOW,YELLOW IN TRACK SECTOR 4
22,2024-03-02T15:19:58+00:00,CLEAR,CLEAR IN TRACK SECTOR 4
66,2024-03-02T16:35:26+00:00,CHEQUERED,CHEQUERED FLAG


In [ ]:
# --- Derive SC / VSC / yellow / recent-red states, then as-of join ---

RED_RECENT_WINDOW = pd.Timedelta(minutes=5)


def build_race_control_states(race_control: pd.DataFrame) -> tuple[pd.DataFrame, list]:
    """
    Walk race_control events in time order and emit a state snapshot after each event.

    Returns
    -------
    states : DataFrame with date + boolean SC/VSC/yellow columns
    red_times : list of timestamps when a red flag was declared
    """
    rc = race_control.copy()
    rc["date"] = pd.to_datetime(rc["date"], utc=True, format="ISO8601")
    rc = rc.sort_values("date")

    sc = vsc = yellow = False
    rows: list[dict] = []
    red_times: list = []

    for r in rc.itertuples(index=False):
        msg = (r.message or "").upper()
        flag = r.flag or ""

        # Safety car / VSC from message text (OpenF1 has no dedicated boolean).
        if "SAFETY CAR DEPLOYED" in msg and "VIRTUAL" not in msg:
            sc = True
        if "SAFETY CAR IN THIS LAP" in msg:
            sc = False
        if "VIRTUAL SAFETY CAR DEPLOYED" in msg or msg.strip() == "VSC DEPLOYED":
            vsc = True
        if "VIRTUAL SAFETY CAR ENDING" in msg or msg.strip() == "VSC ENDING":
            vsc = False

        # Yellow: simplified global on/off (sector clears treated as global clear).
        if flag in ("YELLOW", "DOUBLE YELLOW"):
            yellow = True
        if flag in ("CLEAR", "GREEN", "CHEQUERED"):
            yellow = False

        # Exact / prefix — `"RED FLAG" in msg` falsely matches CHEQUERED FLAG.
        if flag == "RED" or msg.strip() == "RED FLAG" or msg.startswith("RED FLAG"):
            red_times.append(r.date)

        rows.append(
            {
                "date": r.date,
                "safety_car_active": sc,
                "virtual_safety_car_active": vsc,
                "yellow_flag_active": yellow,
            }
        )

    return pd.DataFrame(rows), red_times


def add_race_control_features(driver_laps: pd.DataFrame, race_control: pd.DataFrame) -> pd.DataFrame:
    """Attach race-control flags at end-of-lap as_of (session-wide, no driver key)."""
    out = driver_laps.copy()
    if "as_of" not in out.columns:
        raise ValueError("expected as_of — run the position join first")

    states, red_times = build_race_control_states(race_control)

    joined = pd.merge_asof(
        out.sort_values("as_of"),
        states,
        left_on="as_of",
        right_on="date",
        direction="backward",
    )
    # Laps before the first RC message: treat flags as inactive.
    for col in ("safety_car_active", "virtual_safety_car_active", "yellow_flag_active"):
        joined[col] = joined[col].fillna(False).astype(bool)

    # red_flag_recent: True if a red was declared in the last 5 minutes.
    def _red_recent(t: pd.Timestamp) -> bool:
        return any((t - rd) <= RED_RECENT_WINDOW and rd <= t for rd in red_times)

    joined["red_flag_recent"] = joined["as_of"].map(_red_recent)
    return joined.drop(columns=["date"]).sort_values(["driver_number", "lap_number"]).reset_index(drop=True)


with_rc = add_race_control_features(with_weather, race_control)

print(
    "SC/VSC/yellow/red True counts:",
    with_rc["safety_car_active"].sum(),
    with_rc["virtual_safety_car_active"].sum(),
    with_rc["yellow_flag_active"].sum(),
    with_rc["red_flag_recent"].sum(),
)
with_rc.loc[
    with_rc["driver_number"] == 1,
    ["lap_number", "yellow_flag_active", "safety_car_active", "virtual_safety_car_active", "red_flag_recent"],
].iloc[7:12]


## Pit-window labels

These are **training targets**, not live-known features. They look **forward** in time.

At the end of completed lap `L`, using that driver’s pit laps:

```text
pit_within_N_laps = 1  if there is a pit with  L < pit_lap ≤ L+N
                  = 0  otherwise
```

So on the pit lap itself (e.g. 17), the label is about the **next** stop — usually 0 for short windows.

We reuse the `pits` DataFrame from the earlier pit-join cell (re-run that cell if needed).

**Not yet:** `next_compound` (separate step).


In [76]:
# --- Forward-looking pit labels (3 / 5 / 7 lap windows) ---
# Requires `pits` from the pit-load cell and `with_rc` as the current main table.


def add_pit_window_labels(driver_laps: pd.DataFrame, pits: pd.DataFrame) -> pd.DataFrame:
    """
    Attach pit_within_3/5/7_laps (0/1) using future pits only.

    For lap L: label is 1 iff some pit_lap satisfies L < pit_lap <= L+N.
    """
    out = driver_laps.copy()

    # Sorted pit laps per driver for fast window checks.
    pits_by_driver = {
        driver: group["lap_number"].sort_values().to_numpy()
        for driver, group in pits.groupby("driver_number")
    }

    within_3: list[int] = []
    within_5: list[int] = []
    within_7: list[int] = []

    for row in out.itertuples(index=False):
        lap = row.lap_number
        driver_pits = pits_by_driver.get(row.driver_number, [])

        # Future stops only — the stop on lap L is already "in the past" for labels.
        within_3.append(int(any(lap < p <= lap + 3 for p in driver_pits)))
        within_5.append(int(any(lap < p <= lap + 5 for p in driver_pits)))
        within_7.append(int(any(lap < p <= lap + 7 for p in driver_pits)))

    out["pit_within_3_laps"] = within_3
    out["pit_within_5_laps"] = within_5
    out["pit_within_7_laps"] = within_7
    return out


with_labels = add_pit_window_labels(with_rc, pits)

print("positive rates 3/5/7:",
      with_labels["pit_within_3_laps"].mean().round(3),
      with_labels["pit_within_5_laps"].mean().round(3),
      with_labels["pit_within_7_laps"].mean().round(3))
with_labels.loc[
    with_labels["driver_number"] == 1,
    ["lap_number", "number_of_pit_stops", "pit_within_3_laps", "pit_within_5_laps", "pit_within_7_laps"],
].iloc[12:20]


positive rates 3/5/7: 0.112 0.186 0.26


,lap_number,number_of_pit_stops,pit_within_3_laps,pit_within_5_laps,pit_within_7_laps
12,13,0,0,1,1
13,14,0,1,1,1
14,15,0,1,1,1
15,16,0,1,1,1
16,17,1,0,0,0
17,18,1,0,0,0
18,19,1,0,0,0
19,20,1,0,0,0


## Label: `next_compound`

Yes — this comes from **stints**, not from inventing compounds.

Each stint has a `compound`. For a lap on stint `n`:

```text
next_compound = compound of stint n+1   (same driver)
              = null                    if this is the driver's last stint
```

That is the tyre set they will take at their **next** stop (training target for the multiclass compound model).

Requires `stints` and `current_stint_number` (from the stint-join cells).


In [77]:
# --- next_compound label from successive stints ---
# Uses `stints` (loaded earlier) and current_stint_number on with_labels.


def add_next_compound_label(driver_laps: pd.DataFrame, stints: pd.DataFrame) -> pd.DataFrame:
    """
    Attach next_compound: compound of the driver's following stint, else null.

    This is a label (uses future stint info). Do not treat it as a live feature.
    """
    out = driver_laps.copy()

    # Map (driver, stint_number) → compound of stint_number + 1.
    rows: list[dict] = []
    for driver, group in stints.groupby("driver_number"):
        ordered = group.sort_values("stint_number")
        compound_by_stint = dict(zip(ordered["stint_number"], ordered["compound"]))
        for stint_num in ordered["stint_number"]:
            rows.append(
                {
                    "driver_number": driver,
                    "current_stint_number": stint_num,
                    "next_compound": compound_by_stint.get(stint_num + 1),  # None if last
                }
            )

    lookup = pd.DataFrame(rows)
    return out.merge(lookup, on=["driver_number", "current_stint_number"], how="left")


with_all_labels = add_next_compound_label(with_labels, stints)

print("next_compound null rate:", with_all_labels["next_compound"].isna().mean().round(3))
print("value counts:\n", with_all_labels["next_compound"].value_counts(dropna=False))
with_all_labels.loc[
    with_all_labels["driver_number"] == 1,
    ["lap_number", "current_stint_number", "current_compound", "next_compound", "pit_within_3_laps"],
].iloc[[0, 15, 16, 17, 35, 36, 37, 56]]


next_compound null rate: 0.389
value counts:
 next_compound
HARD    579
None    439
SOFT    111
Name: count, dtype: int64


,lap_number,current_stint_number,current_compound,next_compound,pit_within_3_laps
0,1,1,SOFT,HARD,0
15,16,1,SOFT,HARD,1
16,17,1,SOFT,HARD,0
17,18,2,HARD,SOFT,0
35,36,2,HARD,SOFT,1
36,37,2,HARD,SOFT,0
37,38,3,SOFT,None,0
56,57,3,SOFT,None,0


## Compound-history flags

Leakage-safe: only stints that have **already started** by lap `L` (`lap_start ≤ L`).

From those compounds:

| Column | Meaning |
|--------|---------|
| `compounds_used_so_far` | count of distinct compounds so far |
| `has_used_soft` … `has_used_wet` | boolean flags |
| `has_used_two_dry_compounds` | ≥2 of {SOFT, MEDIUM, HARD} |
| `previous_compound` | compound of prior stint (null on stint 1) |

Bahrain: nobody used MEDIUM — `has_used_medium` stays False; after the first stop `has_used_two_dry_compounds` becomes True (SOFT+HARD).


In [78]:
# --- Compound history from stints started so far (no future stints) ---

DRY_COMPOUNDS = frozenset({"SOFT", "MEDIUM", "HARD"})


def add_compound_history_features(driver_laps: pd.DataFrame, stints: pd.DataFrame) -> pd.DataFrame:
    """
    Attach has_used_* flags, compounds_used_so_far, previous_compound.

    Only counts stints with lap_start <= current lap (already begun).
    """
    out = driver_laps.copy()

    # Pre-sort each driver's stints once.
    stints_by_driver = {
        driver: group.sort_values("stint_number")
        for driver, group in stints.groupby("driver_number")
    }

    records: list[dict] = []
    for row in out.itertuples(index=False):
        started = stints_by_driver[row.driver_number]
        started = started[started["lap_start"] <= row.lap_number]
        compounds = list(started["compound"])
        used = set(compounds)

        records.append(
            {
                "compounds_used_so_far": len(used),
                "has_used_soft": "SOFT" in used,
                "has_used_medium": "MEDIUM" in used,
                "has_used_hard": "HARD" in used,
                "has_used_intermediate": "INTERMEDIATE" in used,
                "has_used_wet": "WET" in used,
                "has_used_two_dry_compounds": len(used & DRY_COMPOUNDS) >= 2,
                # Prior stint's compound; None on the opening stint.
                "previous_compound": compounds[-2] if len(compounds) >= 2 else None,
            }
        )

    return pd.concat([out.reset_index(drop=True), pd.DataFrame(records)], axis=1)


with_history = add_compound_history_features(with_all_labels, stints)

with_history.loc[
    with_history["driver_number"] == 1,
    [
        "lap_number",
        "current_compound",
        "previous_compound",
        "compounds_used_so_far",
        "has_used_soft",
        "has_used_medium",
        "has_used_hard",
        "has_used_two_dry_compounds",
    ],
].iloc[[0, 16, 17, 36, 37, 56]]


,lap_number,current_compound,previous_compound,compounds_used_so_far,has_used_soft,has_used_medium,has_used_hard,has_used_two_dry_compounds
0,1,SOFT,None,1,True,False,False,False
16,17,SOFT,None,1,True,False,False,False
17,18,HARD,SOFT,2,True,False,True,True
36,37,HARD,SOFT,2,True,False,True,True
37,38,SOFT,HARD,2,True,False,True,True
56,57,SOFT,HARD,2,True,False,True,True


## Pace rollups (trailing only)

From `laps.json` `lap_duration` (and sector times). **No future laps.**

| Column | Rule |
|--------|------|
| `current_lap_time` | this lap’s `lap_duration` |
| `previous_lap_time` | prior lap (same driver); null on lap 1 |
| `rolling_mean_lap_time_3/5` | mean of last 3/5 laps **including current** |
| `rolling_median_lap_time_3` | median of last 3 including current |
| `pace_delta_to_recent_average` | current − 3-lap mean |
| `pace_delta_to_stint_best` | current − best time in this stint so far |
| `duration_sector_1/2/3` | from laps |
| `is_pit_out_lap` | from laps (out-laps are often ~20s slower) |

Pit-out laps (e.g. driver 1 lap **18**) will spike the rolling mean — expected.


In [79]:
# --- Trailing pace features from lap times (past + current only) ---


def add_pace_features(driver_laps: pd.DataFrame, laps: pd.DataFrame) -> pd.DataFrame:
    """Attach current/previous/rolling pace cols and sector times; no future leakage."""
    out = driver_laps.copy()

    # Bring sector times + pit-out flag (lap_duration may already be present).
    extra = laps[
        [
            "driver_number",
            "lap_number",
            "lap_duration",
            "duration_sector_1",
            "duration_sector_2",
            "duration_sector_3",
            "is_pit_out_lap",
        ]
    ]
    if "lap_duration" in out.columns:
        out = out.drop(columns=["lap_duration"])
    out = out.merge(extra, on=["driver_number", "lap_number"], how="left")
    out = out.sort_values(["driver_number", "lap_number"])

    g = out.groupby("driver_number", group_keys=False)

    out["current_lap_time"] = out["lap_duration"]
    out["previous_lap_time"] = g["lap_duration"].shift(1)

    # Rolling windows include the current lap (min_periods=1 for early race).
    out["rolling_mean_lap_time_3"] = g["lap_duration"].transform(
        lambda s: s.rolling(3, min_periods=1).mean()
    )
    out["rolling_mean_lap_time_5"] = g["lap_duration"].transform(
        lambda s: s.rolling(5, min_periods=1).mean()
    )
    out["rolling_median_lap_time_3"] = g["lap_duration"].transform(
        lambda s: s.rolling(3, min_periods=1).median()
    )
    out["pace_delta_to_recent_average"] = (
        out["current_lap_time"] - out["rolling_mean_lap_time_3"]
    )

    # Best lap in this stint so far (expanding min within driver+stint).
    stint_g = out.groupby(["driver_number", "current_stint_number"], group_keys=False)
    stint_best = stint_g["lap_duration"].transform(lambda s: s.expanding().min())
    out["pace_delta_to_stint_best"] = out["lap_duration"] - stint_best

    return out.reset_index(drop=True)


with_pace = add_pace_features(with_history, laps)

with_pace.loc[
    with_pace["driver_number"] == 1,
    [
        "lap_number",
        "current_lap_time",
        "previous_lap_time",
        "rolling_mean_lap_time_3",
        "pace_delta_to_recent_average",
        "pace_delta_to_stint_best",
        "is_pit_out_lap",
    ],
].iloc[[0, 1, 2, 16, 17, 18]].round(3)


,lap_number,current_lap_time,previous_lap_time,rolling_mean_lap_time_3,pace_delta_to_recent_average,pace_delta_to_stint_best,is_pit_out_lap
0,1,97.759,NaN,97.759,0.000,0.000,False
1,2,96.296,97.759,97.028,-0.731,0.000,False
2,3,96.753,96.296,96.936,-0.183,0.457,False
16,17,99.896,97.168,98.025,1.871,3.600,False
17,18,117.854,99.896,104.973,12.881,0.000,True
18,19,95.283,117.854,104.344,-9.061,0.000,False


## Save CSV

Write `with_pace` to `data/processed/` (gitignored — local artifact only).

Filename mirrors the session folder so later races do not overwrite each other.


In [80]:
# --- Persist the Bahrain driver-lap table ---
# Requires with_pace from the pace cell (Run All if needed).

OUT_DIR = ROOT / "data" / "processed" / "2024"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Same session id as the raw folder, plus a clear table suffix.
OUT_PATH = OUT_DIR / "1229_9472_bahrain_race_driver_laps.csv"

with_pace.to_csv(OUT_PATH, index=False)

print("wrote:", OUT_PATH)
print("rows:" , len(with_pace), "cols:", with_pace.shape[1])
print("size_mb:", round(OUT_PATH.stat().st_size / 1e6, 3))


wrote: /home/moth/Desktop/Formula-1-Live-Strategy-Tool/data/processed/2024/1229_9472_bahrain_race_driver_laps.csv
rows: 1129 cols: 49
size_mb: 0.373


## Stop here

After running the save cell you should see a file like:

`data/processed/2024/1229_9472_bahrain_race_driver_laps.csv`

Open it or reload with `pd.read_csv(...)` to spot-check columns.

**Next (when ready):** extract reusable `process_race(session_dir)` so we can loop other races — do not train yet.
